# Has Streaming Made Hit Songs Shorter? — Data preparation
DS4200 group project. This notebook turns the two raw files into the tables the charts use.

**Inputs** (in `data/raw/`)
- `spotify_data.csv` (in `data/raw/` or the parent project folder): Spotify 1M Tracks (Kaggle, scraped June 2023), 1,159,764 tracks from 2000–2023
- `hot-100-current.csv`: Billboard Hot 100 weekly chart history from `utdata/rwd-billboard-data` on GitHub (downloaded automatically if missing)

**Outputs** (in `data/processed/`)
- `hits_clean.csv`: one row per Hot 100 song that debuted 2000–2023, joined to Spotify where possible
- `all_music_yearly.csv`: the "all music" baseline, yearly median and interquartile range of track length from the full Spotify file
- `genre_family_yearly.csv`: median track length by genre family and year from the full Spotify file

Steps follow the analysis plan in Document 2.

In [1]:
import os, re, unicodedata, urllib.request
import numpy as np
import pandas as pd

RAW, OUT = "data/raw", "data/processed"
# Spotify CSV: data/raw/ or the project folder one level up (too big for GitHub, so it's gitignored)
SPOTIFY = next(p for p in [f"{RAW}/spotify_data.csv", "../spotify_data.csv"] if os.path.exists(p))
os.makedirs(OUT, exist_ok=True); os.makedirs(RAW, exist_ok=True)
BB_URL = "https://raw.githubusercontent.com/utdata/rwd-billboard-data/main/data-out/hot-100-current.csv"
if not os.path.exists(f"{RAW}/hot-100-current.csv"):
    urllib.request.urlretrieve(BB_URL, f"{RAW}/hot-100-current.csv")

## 1. Text keys for matching
Billboard and Spotify spell things differently ("Jay-Z" vs "JAY-Z", "Song (feat. X)" vs "Song", "Song - Remastered 2011").
Titles and artists are lowercased, stripped of accents and punctuation, and cleared of feature credits and version suffixes.
Billboard credits every artist in one string ("Lil Baby & Gunna Featuring Drake"), so it is split into individual names.

In [2]:
def norm(s):
    """lowercase, strip accents, drop punctuation, collapse spaces"""
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def clean_title(t):
    """title key: drops ' - Remastered', (feat. ...), [..], trailing 'feat. X'"""
    if pd.isna(t):
        return ""
    t = str(t)
    t = re.split(r"\s+-\s+", t)[0]
    t = re.sub(r"\((feat|ft|with|from)[^)]*\)", "", t, flags=re.I)
    t = re.sub(r"\[[^\]]*\]", "", t)
    t = re.sub(r"\s(feat\.?|ft\.?)\s.*$", "", t, flags=re.I)
    k = norm(t)
    return k if k else norm(re.sub(r"[()]", "", str(t)))

def loose_title(t):
    """looser key for a second pass: drops every parenthetical, e.g. (Taylor's Version)"""
    return norm(re.sub(r"\([^)]*\)", "", re.split(r"\s+-\s+", str(t))[0]))

ARTIST_SPLIT = r"\s+(?:featuring|feat\.?|ft\.?|with|x|and|&|vs\.?|duet with)\s+|,\s*|\s*/\s*"

def performer_artists(p):
    """every artist credited in a Billboard performer string, lead artist first"""
    if pd.isna(p):
        return []
    return [norm(x) for x in re.split(ARTIST_SPLIT, str(p), flags=re.I) if norm(x)]

## 2. Billboard → one row per song
The weekly file has one row per song per week. `peak_pos` and `wks_on_chart` are running totals, so a song's
minimum peak and maximum weeks summarize its whole run. A song is identified by (title, performer), and its year is the year it **debuted**.

In [3]:
bb = pd.read_csv(f"{RAW}/hot-100-current.csv", parse_dates=["chart_week"])
songs = (bb.groupby(["title", "performer"], as_index=False)
           .agg(debut_week=("chart_week", "min"),
                peak_pos=("peak_pos", "min"),
                weeks_on_chart=("wks_on_chart", "max")))
songs["debut_year"] = songs.debut_week.dt.year
songs = songs[songs.debut_year.between(2000, 2023)].reset_index(drop=True)
songs["top10"] = songs.peak_pos <= 10
songs["number_one"] = songs.peak_pos == 1
songs["title_key"] = songs.title.map(clean_title)
songs["artists"] = songs.performer.map(performer_artists)
songs["lead_key"] = songs.artists.str[0]
songs["song_id"] = np.arange(len(songs))
print(f"{len(bb):,} weekly rows -> {len(songs):,} songs that debuted 2000-2023")

356,100 weekly rows -> 10,432 songs that debuted 2000-2023


## 3. Join to Spotify
**Pass 1:** exact match on the cleaned title plus any credited artist. Spotify lists one artist per track, and it is sometimes a featured artist rather than the lead.

**Version choice:** about a quarter of matched songs have several Spotify versions (single, album, remaster, deluxe). The rules prefer
a lead-artist match first, then the release year closest to the chart debut, then the most popular version.

**Pass 2:** for songs still unmatched, the same match on the looser title with every parenthetical removed. It is limited to Spotify releases within 2 years of the debut, so a 2023 re-recording is not paired with the 2010 original.

In [4]:
sp = pd.read_csv(SPOTIFY, index_col=0)
sp = sp.dropna(subset=["artist_name", "track_name"])        # 16 rows missing a name
sp["title_key"] = sp.track_name.map(clean_title)
sp["loose_key"] = sp.track_name.map(loose_title)
sp["artist_key"] = sp.artist_name.map(norm)

cand = (songs[["song_id", "title", "title_key", "artists", "lead_key", "debut_year"]]
        .explode("artists").rename(columns={"artists": "artist_key"}))
cand["loose_key"] = cand.title.map(loose_title)

def pick_best(m):
    m = m.assign(lead_match=m.artist_key == m.lead_key,
                 yr_gap=(m.year - m.debut_year).abs())
    m = m.sort_values(["song_id", "lead_match", "yr_gap", "popularity"],
                      ascending=[True, False, True, False])
    return m.drop_duplicates("song_id")

m1 = cand.merge(sp.drop(columns="loose_key"), on=["title_key", "artist_key"])
best1 = pick_best(m1).assign(match_pass=1)
multi = m1.groupby("song_id").track_id.nunique().gt(1).mean()

rest = cand[~cand.song_id.isin(best1.song_id)]
m2 = rest.drop(columns="title_key").merge(sp.drop(columns="title_key"), on=["loose_key", "artist_key"])
m2 = m2[(m2.loose_key != "") & ((m2.year - m2.debut_year).abs() <= 2)]
best2 = pick_best(m2).assign(match_pass=2)

best = pd.concat([best1, best2])
print(f"pass 1: {len(best1):,} | pass 2: +{len(best2):,} | {multi:.0%} of pass-1 matches had several versions")

pass 1: 7,632 | pass 2: +147 | 24% of pass-1 matches had several versions


## 4. Assemble the hits table and check coverage
Unmatched songs stay in the table with `matched = False` so Billboard-only analysis can still use them.
The Spotify file was scraped in June 2023, so songs that debuted after that **cannot** match. They are flagged with `after_spotify_scrape`.

In [5]:
spot_cols = ["track_id", "artist_name", "track_name", "year", "genre", "popularity", "duration_ms",
             "danceability", "energy", "valence", "tempo", "loudness", "speechiness", "acousticness"]
hits = songs.merge(best[["song_id", "match_pass"] + spot_cols], on="song_id", how="left")
hits = hits.rename(columns={"year": "spotify_year"})
hits["matched"] = hits.track_id.notna()
hits["release_gap_yrs"] = hits.spotify_year - hits.debut_year
hits["after_spotify_scrape"] = hits.debut_week > "2023-06-30"

usable = hits[~hits.after_spotify_scrape]
print(f"overall match rate: {hits.matched.mean():.1%}  |  excluding post-scrape debuts: {usable.matched.mean():.1%}")
hits.groupby("debut_year").matched.mean().round(2).to_frame("share matched").T

overall match rate: 74.6%  |  excluding post-scrape debuts: 77.2%


debut_year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
share matched,0.69,0.78,0.82,0.85,0.86,0.85,0.84,0.84,0.81,0.86,...,0.86,0.82,0.77,0.84,0.78,0.72,0.63,0.6,0.69,0.2


Coverage is 80–87% for 2002–2015 and falls to about 60% in 2020–2021. Several major artists are missing from the Spotify file entirely (Post Malone, Kanye West, Bad Bunny),
so **matched hits under-represent recent hip-hop and Latin music**. Charts that use Spotify fields should say "matched hits only."
The planned fix is to fill lengths for unmatched songs from a second source.

In [6]:
missing_artists = (hits[~hits.matched & ~hits.after_spotify_scrape]
                   .performer.str.split(r" Featuring | & |, ", regex=True).str[0]
                   .value_counts().head(10))
missing_artists

performer
YoungBoy Never Broke Again    88
Kanye West                    87
Post Malone                   56
Lil Uzi Vert                  52
Juice WRLD                    51
Rod Wave                      49
Bad Bunny                     48
Polo G                        32
Pop Smoke                     30
Metro Boomin                  29
Name: count, dtype: int64

## 5. Derived attributes
- **duration_min**, and **duration_band** (ordinal): under 2:30, 2:30–3:30, over 3:30
- **era** (ordinal): Pre-streaming 2000–11, Early streaming 2012–15, Streaming majority 2016–19, Short-form era 2020–23
- **genre_family**: Spotify's 82 genre labels condensed into 7 families
- **chart_outcome** (ordinal): Top 10, 11–40, 41–100 by peak position

In [7]:
FAMILIES = {
    "Pop": ["pop", "indie-pop", "power-pop", "k-pop", "cantopop", "pop-film"],
    "Hip-Hop & R&B": ["hip-hop", "soul", "funk", "gospel", "groove"],
    "Rock & Metal": ["rock", "alt-rock", "hard-rock", "psych-rock", "punk", "punk-rock", "rock-n-roll",
                     "emo", "goth", "grindcore", "hardcore", "heavy-metal", "metal", "metalcore",
                     "black-metal", "death-metal", "industrial", "ska", "garage", "guitar"],
    "Electronic & Dance": ["dance", "edm", "electro", "electronic", "house", "deep-house", "chicago-house",
                           "progressive-house", "techno", "detroit-techno", "minimal-techno", "trance",
                           "dubstep", "drum-and-bass", "breakbeat", "hardstyle", "club", "party", "disco",
                           "trip-hop", "dub", "chill"],
    "Country & Folk": ["country", "folk", "acoustic", "singer-songwriter", "songwriter"],
    "Latin & Global": ["afrobeat", "dancehall", "forro", "salsa", "samba", "sertanejo", "tango",
                       "spanish", "french", "german", "swedish", "indian"],
    "Jazz, Classical & Other": ["jazz", "blues", "classical", "opera", "piano", "new-age", "ambient",
                                "sleep", "show-tunes", "comedy", "romance", "sad"],
}
GENRE_TO_FAMILY = {g: f for f, gs in FAMILIES.items() for g in gs}
assert set(GENRE_TO_FAMILY) == set(sp.genre.unique()), "every Spotify genre must map to a family"

ERAS = ["Pre-streaming (2000-11)", "Early streaming (2012-15)",
        "Streaming majority (2016-19)", "Short-form era (2020-23)"]
def era(y):
    return ERAS[0] if y <= 2011 else ERAS[1] if y <= 2015 else ERAS[2] if y <= 2019 else ERAS[3]

def add_derived(df, year_col):
    df["duration_min"] = df.duration_ms / 60000
    df["duration_band"] = pd.cut(df.duration_min, [0, 2.5, 3.5, np.inf], right=False,
                                 labels=["Under 2:30", "2:30-3:30", "Over 3:30"])
    df["era"] = df[year_col].map(era)
    df["genre_family"] = df.genre.map(GENRE_TO_FAMILY)
    return df

hits = add_derived(hits, "debut_year")
hits["chart_outcome"] = pd.cut(hits.peak_pos, [0, 10, 40, 100], labels=["Top 10", "11-40", "41-100"])
sp = add_derived(sp, "year")
hits[hits.matched].genre_family.value_counts()

genre_family
Electronic & Dance         2252
Hip-Hop & R&B              1931
Country & Folk             1538
Pop                        1236
Rock & Metal                677
Jazz, Classical & Other     121
Latin & Global               24
Name: count, dtype: int64

Latin & Global (24 matched hits) and Jazz, Classical & Other (about 120) are too small to chart for hits by year. Genre-by-year charts use the full Spotify file, or pool those two families.

## 6. "All music" baselines from the full Spotify file
These are stored as yearly summaries rather than a second sample of tracks. The Spotify file caps each genre at roughly 1,000 tracks a year, so **track counts reflect the collection process, not real release volume**. Medians and quartiles can be compared across years. Counts cannot.

In [8]:
q25 = lambda s: s.quantile(.25)
q75 = lambda s: s.quantile(.75)
all_music = (sp.groupby("year").duration_min
               .agg(median="median", q25=q25, q75=q75, n_tracks="size").reset_index())
genre_family_yearly = (sp.groupby(["genre_family", "year"]).duration_min
                         .agg(median="median", q25=q25, q75=q75, n_tracks="size").reset_index())
all_music.round(3)

,year,median,q25,q75,n_tracks
0,2000,3.899,3.048,4.994,44034
1,2001,3.907,3.089,4.990,42351
2,2002,3.943,3.119,5.043,42102
3,2003,3.894,3.064,5.004,42368
4,2004,3.917,3.104,4.974,43390
5,2005,3.906,3.074,5.037,43779
6,2006,3.935,3.132,5.086,45474
7,2007,3.946,3.146,5.110,46001
8,2008,3.931,3.142,5.098,47432
9,2009,3.943,3.146,5.107,46869


## 7. Save

In [9]:
hit_cols = ["song_id", "title", "performer", "debut_week", "debut_year", "era", "peak_pos", "weeks_on_chart",
            "top10", "number_one", "chart_outcome", "matched", "match_pass", "after_spotify_scrape",
            "track_id", "artist_name", "track_name", "spotify_year", "release_gap_yrs",
            "genre", "genre_family", "popularity", "duration_ms", "duration_min", "duration_band",
            "danceability", "energy", "valence", "tempo", "loudness", "speechiness", "acousticness"]
hits[hit_cols].to_csv(f"{OUT}/hits_clean.csv", index=False)
all_music.to_csv(f"{OUT}/all_music_yearly.csv", index=False)
genre_family_yearly.to_csv(f"{OUT}/genre_family_yearly.csv", index=False)
print(f"hits_clean.csv: {len(hits):,} rows x {len(hit_cols)} cols ({hits.matched.sum():,} matched to Spotify)")

hits_clean.csv: 10,432 rows x 32 cols (7,779 matched to Spotify)
